In [4]:
# https://qwen.readthedocs.io/en/latest/framework/LlamaIndex.html

from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


def completion_to_prompt(completion):
    return f"<|im_start|>system\n<|im_end|>\n<|im_start|>user\n{completion}<|im_end|>\n<|im_start|>assistant\n"


def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == "system":
            prompt += f"<|im_start|>system\n{message.content}<|im_end|>\n"
        elif message.role == "user":
            prompt += f"<|im_start|>user\n{message.content}<|im_end|>\n"
        elif message.role == "assistant":
            prompt += f"<|im_start|>assistant\n{message.content}<|im_end|>\n"

    if not prompt.startswith("<|im_start|>system"):
        prompt = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n" + prompt

    prompt = prompt + "<|im_start|>assistant\n"

    return prompt


def setup(model, tokenizer):
    Settings.llm = HuggingFaceLLM(
        model_name=model,
        tokenizer_name=tokenizer,
        context_window=40960,
        max_new_tokens=8000,
        generate_kwargs={
            "temperature": 0.7,
            "top_k": 20,
            "top_p": 0.8,
        },  # {"temperature": 0.7, "top_k": 50, "top_p": 0.95},
        messages_to_prompt=messages_to_prompt,
        completion_to_prompt=completion_to_prompt,
        device_map="auto",
    )

def create_save_vsidx(save_dir, embed_model, input_dir = None, input_files = None):
    Settings.embed_model = HuggingFaceEmbedding(
        model_name=embed_model,
    )

    Settings.transformations = [SentenceSplitter(chunk_size=512)]

    from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

    if input_dir is not None:
        documents = SimpleDirectoryReader(
            input_dir=input_dir, recursive=True
        ).load_data()
    elif input_files is not None:
        documents = SimpleDirectoryReader(
            input_files=input_files
        ).load_data()

    print(f"len(documents)={len(documents)}\n")

    Settings.index = VectorStoreIndex.from_documents(
        documents,
        embed_model=Settings.embed_model,
        transformations=Settings.transformations,
        show_progress=True
    )
    Settings.index.storage_context.persist(persist_dir=save_dir)

def load_vsidx(save_dir, embed_model):
    Settings.embed_model = HuggingFaceEmbedding(
        model_name=embed_model,
    )

    from llama_index.core import StorageContext, load_index_from_storage
    storage_context = StorageContext.from_defaults(persist_dir=save_dir)

    Settings.index = load_index_from_storage(storage_context)

def query(query_text):
    query_engine = Settings.index.as_query_engine(streaming=True, similarity_top_k=30)
    r = query_engine.query(query_text)
    r.print_response_stream()

In [5]:
# run once to create vector store
save_dir="alldocuments"

import os
if os.path.isdir(save_dir):
    pass
else:
    create_save_vsidx(save_dir=save_dir, embed_model="BAAI/bge-m3", input_dir="../document" )

In [6]:
setup(model = "Qwen/Qwen3-14B-FP8", tokenizer = "Qwen/Qwen3-0.6B-FP8")
#setup(model = "Qwen/Qwen2.5-7B-Instruct", tokenizer = "Qwen/Qwen2.5-7B-Instruct")

load_vsidx(save_dir=save_dir, embed_model = "BAAI/bge-m3")



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The model `Qwen/Qwen3-14B-FP8` and tokenizer `Qwen/Qwen3-0.6B-FP8` are different, please ensure that they are compatible.


In [7]:
query("周の共王の事績について教えてください。")
query("周の懿王の事績について教えてください。")
query("周の孝王の事績について教えてください。")
query("周の夷王の事績について教えてください。")


<think>
Okay, let's see. The user is asking about the achievements of King Gongwang of Zhou. I need to look through the provided context information to find any mentions of him.

Looking through the files, I see several entries from the "周本紀" (Annals of Zhou) in the Shiji (Records of the Grand Historian). There's a section that mentions Gongwang. Let me check that part again.

Here's a relevant excerpt: "共王遊於涇上，密康公從，有三女?之。其母曰：「必致之王。夫獸三為群，人三為眾，女三為粲。王田不取群，公行不下眾，王禦不參一族。夫粲，美之物也。眾以美物歸女，而何德以堪之？王猶不堪，況爾之小丑乎！小丑備物，終必亡。」康公不獻，一年，共王滅密。"

So, the context says that King Gongwang was touring by the泾 River, and Mi Kanggong accompanied him. There were three women who were beautiful, and their mother advised Mi Kanggong to present them to the king. The mother's reasoning was based on the idea that three of a kind (like three animals, three people, three beautiful women) should be given to the king. If the king doesn't take them, it's a sign of his virtue, but if a minor official like Mi Kanggong keeps th

In [4]:
query("戦国時代の斉の田単の事績について教えてください。")

<think>
Okay, let's tackle this query about the achievements of Tian Dan (田單) from the Warring States period. The user provided several documents from the Shiji (Records of the Grand Historian) and Zizhi Tongjian (Comprehensive Mirror to Aid in Government), so I need to extract relevant information from those.

First, I'll look for mentions of Tian Dan in the Shiji. In the "Tian Dan Biography" (田單列傳), there's a story about how he saved the state of Qi. When the Yan army invaded, Tian Dan was in Anping and had his relatives attach iron rings to the axles of their carts. This helped them escape when the city fell. Later, he became the general of Jimo and resisted the Yan forces.

There's a part about the "Fire牛" (火牛) strategy. He used decorated cows with burning tails to charge the Yan army, causing chaos. This, combined with a surprise attack, led to the Yan army's defeat. After this victory, Tian Dan helped restore the Qi state, and the king was brought back from Ju. He was then grante

In [5]:
query("越王勾践と范蠡について教えてください。")

<think>
Okay, I need to answer the question about King Goujian of Yue and Fan Li based on the provided context. Let me start by going through the documents to gather relevant information.

First, from the "史記_越王句踐世家" texts, I see that King Goujian was the ruler of Yue who faced defeat by Wu, was humiliated at the Battle of Boju, and later avenged his defeat by conquering Wu. He worked closely with Fan Li, who was a key advisor. Fan Li helped him plan the revenge, and after the success, he advised Goujian to let him go to avoid suspicion. Fan Li then went on to become a wealthy merchant, known as Tao Zhu Gong.

There's mention of Goujian's strategies, like the "卧薪尝胆" (sleeping on brushwood and tasting gall) to remember his humiliation. He also sent Fan Li to serve as a hostage in Wu, which led to his eventual release. After conquering Wu, Goujian became a hegemon, but Fan Li foresaw danger and left, warning that "when the birds are caught, the bow is put away," which later proved true w

In [6]:
query("漢武帝の封禅について教えてください。")

<think>
Okay, I need to answer the question about Emperor Wu of Han's Fengchan (封禅). Let me look through the provided context information to gather relevant details.

First, I see mentions of Emperor Wu discussing Fengchan with officials and scholars. There's a part where after obtaining a bronze tripod (寶鼎), he consults with his ministers and scholars about Fengchan. The text says that the rituals were rare and not well-known, so they looked into ancient texts like the Book of Documents, Zhou Officials, and Wang Zhi to find references to sacrifices and rituals involving shooting cattle. A man named Ding Gong from Qi, over ninety years old, mentions that Fengchan is about achieving immortality, and that Qin Shi Huang couldn't ascend the mountain, but the emperor should try. 

Then, the emperor orders scholars to practice shooting cattle and draft the Fengchan rituals. After several years, they prepare to proceed. However, when the emperor hears from Gong Sun Qing and the alchemists tha

In [7]:
query("斉の桓公の晩年について教えてください。")

<think>
Okay, let's tackle this query about the late years of Duke Huan of Qi. The user provided several documents, mostly from Chinese historical texts like the "Spring and Autumn Annals" and their commentaries, as well as the "Records of the Grand Historian" (Shiji) and "Zizhi Tongjian." 

First, I need to find mentions of Duke Huan of Qi's later years. Looking through the documents, there's a section in the Shiji (史記_齊太公世家) that talks about his death and the succession issues. It mentions that after Duke Huan's death, there were conflicts among his sons, leading to a power struggle. Specifically, it says that five of his sons fought for the throne, and the palace was left empty, with his body lying unburied for 67 days until his son Wuji was eventually buried.

Another part from the same Shiji mentions that after Duke Huan's death, his son Zhuanggong (Zhuanggong) was eventually installed as the new ruler, but there were conflicts with the other sons. Also, there's a reference to the